In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

### Q1. Write a PySpark query to find the Top 3 most frequent values in a column.

In [0]:
orders_df = spark.read.table("workspace.default.orders")
display(orders_df)

# Solution 1
result_df = orders_df.groupBy("product").count().orderBy("count", ascending=False).limit(3)
display(result_df)

# Solution 2:- Using the functions API to sort inside a column expression directly
result_df = (orders_df
             .groupBy("product")
             .agg(F.count("*").alias("total_count"))
             .orderBy(F.col("total_count").desc())
             .limit(3))
display(result_df)

#### 🏗️ How This Query Executes Across a Spark Cluster

Because Spark is a distributed compute engine, your data is chopped up into multiple **partitions** across different worker nodes. Calculating a top 5 list requires moving data across the network, which triggers a **Shuffle**.

##### Phase 1: Local Aggregation (Map Side)

* **What happens:** Each worker node reads its assigned data partition from cloud storage.
* **Optimization:** Before sending data over the network, Spark performs a local, partial aggregation on each worker. For example, if Worker 1 sees "Mouse" three times, it summarizes it locally as `(Mouse, 3)` instead of sending three individual rows. This drastically cuts down network traffic.

##### Phase 2: Data Shuffling (Exchange)

* **What happens:** To calculate the absolute total count, all rows sharing the exact same `product` key must be routed to the exact same worker node.
* **The Cost:** This triggers a **Shuffle Exchange**. Data is serialized, pushed across the network cluster, and deserialized on the receiving nodes.

##### Phase 3: Global Aggregation & Sorting (Reduce Side)

* **What happens:** The target worker nodes receive all shuffled keys, compute the final global sum for each product, and sort the rows locally within their respective partitions based on the count descending.

##### Phase 4: The `.limit()` Optimization

* **What happens:** This is where Spark does something incredibly smart. Instead of collecting all millions of sorted records from the entire cluster back to the driver node, **each individual partition calculates its own local Top 3**.
* **The Benefit:** Each worker node throws away everything except its top 3 rows. It then sends *only* those 3 rows to the driver node. The driver node collects these tiny fractions of data, runs a final ultra-fast sort on the handful of rows, and returns the absolute top 3 records to your `display()` window.

---

### Q2. Remove duplicates in PySpark using multiple columns & keep only the latest record.

In [0]:
user_updates_df = spark.read.table("workspace.default.user_updates")
display(user_updates_df)

window_spec = Window.partitionBy("name" , "country").orderBy(F.col("updated_timestamp").desc())

result_df = user_updates_df.withColumn("rank", F.row_number().over(window_spec)).filter(F.col("rank") == 1).drop("rank")
display(result_df)

#### What is the physical difference in cluster resource usage between using your Window approach versus just calling `.dropDuplicates(["name", "country"])`?"*

Here is the exact architectural comparison to document

---

#### Window Functions vs. dropDuplicates() Under the Hood

##### 1. The Window Function Approach (Your Solution)

```python

window_spec = Window.partitionBy("name" , "country").orderBy(F.col("updated_timestamp").desc())

```

* **How it executes:** Spark shuffles the data so that all records sharing the same `name` and `country` land on the same worker node. Once there, Spark performs a **full physical sort** on the `updated_timestamp` column inside every single partition to assign the row numbers sequentially (`1, 2, 3...`).
* **Resource Impact:** Sorting data inside partitions takes extra CPU and memory. If one specific user combination has millions of updates (data skew), that single worker node might run out of memory (OOM error).
* **The Benefit:** 100% deterministic. You know exactly which row you are keeping.

##### 2. The `.dropDuplicates()` Approach

```python

result_df = user_updates_df.dropDuplicates(["name", "country"])

```

* **How it executes:** Spark shuffles the data by the key columns just like the window function. However, instead of sorting the records, Spark uses an internal **HashMap** structure on each worker node. As rows stream past, Spark checks if the `(name, country)` key is already in the map. If it is, the row is instantly dropped.
* **Resource Impact:** Much faster and uses significantly less CPU because **no sorting happens**.
* **The Catch:** Non-deterministic. Spark will just grab the very first record that happens to land on the worker node first. You lose the ability to guarantee you kept the "latest" timestamp.

---

##### 📝 Key Takeaway for Interview Notes:

- Use `.dropDuplicates()` when performance is the priority and any arbitrary row is fine. Use `Window + row_number()` when business logic demands strict accuracy (like keeping the latest record), even if it costs more CPU cycles to sort.

### Q3. How do you handle data skew in PySpark joins?

#### What is Data Skew?

**Data skew** is one of the most common performance bottlenecks in distributed joins. It occurs when certain join keys appear far more frequently than others, causing uneven data distribution across partitions. This leads to some workers processing massive amounts of data while others sit idle, resulting in **straggler tasks**, long job durations, and potential **OOM (Out of Memory) errors**.

---
#### Why Data Skew Happens

During a join, Spark uses **shuffle hash join** by default — all rows with the same join key are routed to the same partition. If 80% of your data shares one key value (like `NULL`, a default value, or a popular product ID), that single partition becomes a bottleneck.

---
#### How do you handle data skew in PySpark joins

#### Strategy 1: Broadcast Join

**When to use:** One table is small enough to fit in memory (typically < 10 MB, configurable).

**How it works:** Spark copies the entire small table to every worker node, eliminating the shuffle entirely. No data movement means skew becomes irrelevant.

```python
from pyspark.sql import functions as F

df_result = large_df.join(F.broadcast(small_df), "key", "inner")
```

**Configuration:** Spark auto-broadcasts tables under the threshold set by `spark.sql.autoBroadcastJoinThreshold` (default 10 MB). You can adjust this if workers have sufficient memory.

---
#### Strategy 2: Salting

**When to use:** Both tables are large and cannot be broadcast, and you know which keys are skewed.

**How it works:** 
1. Add a random suffix (0 to N) to the skewed join key in the large table
2. Replicate each row in the small table N times with all possible suffixes
3. Join on the salted key
4. Remove the salt column

This distributes the skewed key across N partitions instead of one.

```python
# Add salt to large table
large_salted = large_df.withColumn("salt", (F.rand() * 10).cast("int"))
large_salted = large_salted.withColumn("join_key_salted", 
    F.concat(F.col("key"), F.lit("_"), F.col("salt")))

# Replicate small table with all salt values
small_exploded = small_df.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(10)])))
small_exploded = small_exploded.withColumn("join_key_salted", 
    F.concat(F.col("key"), F.lit("_"), F.col("salt")))

# Join and cleanup
result = large_salted.join(small_exploded, "join_key_salted").drop("salt", "join_key_salted")
```

---
#### Strategy 3: Adaptive Query Execution (AQE)

**When to use:** Spark 3.0+ — let Spark handle skew automatically.

**How it works:** AQE detects skewed partitions at runtime and splits them into smaller sub-partitions, replicating the corresponding data from the other side.

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

This is the **easiest and most recommended approach** for Spark 3.x+. No code changes required.

---
#### Strategy 4: Isolate Skewed Keys

**When to use:** You can identify specific skewed keys (e.g., `NULL` values representing guest users).

**How it works:**
1. Filter out the skewed keys from both DataFrames
2. Join the skewed keys separately using broadcast join
3. Join the remaining keys normally
4. Union the two results

```python
# Separate skewed key
skewed_large = large_df.filter(F.col("key").isNull())
skewed_small = small_df.filter(F.col("key").isNull())

# Non-skewed data
normal_large = large_df.filter(F.col("key").isNotNull())
normal_small = small_df.filter(F.col("key").isNotNull())

# Join separately
skewed_result = skewed_large.join(F.broadcast(skewed_small), "key")
normal_result = normal_large.join(normal_small, "key")

# Combine
final_result = skewed_result.union(normal_result)
```

---
#### Strategy 5: Pre-Aggregation

**When to use:** The join is followed by aggregation.

**How it works:** Aggregate before joining to reduce data volume, potentially making broadcast join feasible.

```python
# Instead of: large_df.join(small_df, "key").groupBy("key").agg(...)
# Do: large_df.groupBy("key").agg(...).join(small_df, "key")
```

---
#### Quick Decision Guide

* **Small table?** → Use broadcast join
* **Spark 3.x+?** → Enable AQE first, it handles most cases automatically
* **Specific skewed keys known?** → Isolate and handle separately
* **Both tables huge with unknown skew?** → Try salting
* **Join followed by aggregation?** → Pre-aggregate first

---
#### Detecting Skew

Monitor the Spark UI:
* **Stage Details:** Look for tasks that take significantly longer than others
* **SQL Tab:** Check for skewed partitions in the exchange operators
* **Executors Tab:** One executor processing far more data than others indicates skew
---

#### If AQE does this automatically, why we still being asked to explain it in a technical round?

An interviewer at a firm like Tiger Analytics isn't just checking if you know how to turn on a configuration switch; they are evaluating your depth as an architect.

Here is exactly why knowing these strategies is vital and why it remains a favorite interview target:

---

##### 1. AQE is a Safety Net, Not a Magic Bullet

Adaptive Query Execution is incredible, but it has strict limitations.

* AQE only optimizes data skew **during a shuffle phase** (like a Shuffle Hash Join or a Sort-Merge Join).
* If your data pipeline is crashing with an **Out Of Memory (OOM) error** during the *initial read phase* or during a custom window function operation due to data distribution issues, AQE cannot save you. You must know how to restructure the code manually using patterns like splitting the keys or pre-aggregating.

---

##### 2. The "Black Box" Danger (Debugging Production Failures)

When a critical pipeline fails at 2:00 AM, a junior engineer looks at the error log, sees a generic executor lost failure, and doesn't know what to do because "the configurations are turned on."

A senior Data Engineer knows how to open the **Spark UI**, look at the Stage DAG, identify the exact task partition that took 4 hours while the others took 2 seconds, and instantly recognize it as a data skew issue. Understanding the mechanics of salting and broadcasting allows you to diagnose *why* the automated system failed to optimize it.

---

##### 3. Testing Deep Architectural Awareness

Interviewers use this question to separate engineers who just write syntax from engineers who understand memory and cluster compute dynamics. Proving you know *how* Spark physically moves rows across network sockets to different nodes during a skewed join shows that you write code with cluster efficiency and cost optimization in mind.

---

#### 📝 How to position this in your interview to stand out:

If you get asked this question, the absolute best way to answer it is to structure your response like this:

- "In modern Spark 3.x pipelines, my first line of defense is always enabling **Adaptive Query Execution (AQE)** to let Spark dynamically split skewed partitions at runtime. However, if the skew is severe enough to bypass AQE bounds or cause OOMs before the shuffle, I apply manual architectural patterns. If one table is small, I force a **Broadcast Hash Join**. If both are massive, I implement a **Salting strategy** or use a **Split-and-Union** method to isolate the skewed keys entirely."

### Q4. Read CSV → Transform → Write partitioned Parquet using PySpark.

In [0]:
# Read Azure usage CSV into a Spark DataFrame
azure_usage_df = spark.read.csv("/Volumes/dbacademy/default/tutorials/AzureUsage.csv", header=True , sep="," ,inferSchema= True)
# display(azure_usage_df) 

# Aggregate total cost per ServiceName and ServiceResource
trans_df = azure_usage_df.groupBy("ServiceName","ServiceResource").agg(F.sum("Cost").alias("total_cost"))

# Define window to rank resources by total cost within each service
window_spec = Window.partitionBy("ServiceName").orderBy(F.col("total_cost").desc())

# Calculate running total of cost per service and order by running total descending
trans_df = trans_df.withColumn("running_total", F.sum("total_cost").over(window_spec)).orderBy(F.col("running_total").desc())
display(trans_df)

# Save the result as Parquet (single partition)
trans_df.write.mode("overwrite").format("parquet").save("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF1")

# Save the result as Parquet (repartitioned into 3)
trans_df.repartition(3).write.mode("overwrite").format("parquet").save("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF2")

# Save the result as Parquet partitioned by ServiceName (creates subfolders for each ServiceName)
trans_df.write.mode("overwrite").partitionBy("ServiceName").format("parquet").save("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF3")

In [0]:
print(spark.read.parquet("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF1").count())

print(spark.read.parquet("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF2").count())

print(spark.read.parquet("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF3").count())

#### 🗂️ `repartition()` vs. `partitionBy()` Under the Hood

| Method | What It Does Physically | Why It's Used |
| --- | --- | --- |
| **`.repartition(3)`** | Forces Spark to redistribute the dataset evenly across **3 unspecific data files** in storage. | Used to balance cluster processing sizes or prevent small-file problems. |
| **`.partitionBy("year")`** | Creates physical **sub-directories** in your storage layer based on column values (e.g., `/year=2025/`, `/year=2026/`). | Used for **Predicate Pushdown / Partition Pruning**. Downstream queries checking `WHERE year = 2026` can completely ignore all other folders, saving massive I/O costs. |

---